# 42_03 - Kurva MCC vs fraksi penyerang semua aturan (Paper 4-2)**Tujuan:** grafik kurva **MCC vs fraksi penyerang** untuk kelima aturan agregasisekaligus (f=0/10/20/30%, serangan label-flipping). Memperkuat RQ1 (kerentananFedAvg) dan RQ2 (ketahanan robust-agg) secara VISUAL dalam satu gambar.**Sumber tunggal:** `robust_agg.py` + `../aws/fed_node.py`.**Output:** `fed_out42/vuln_curve.csv` + `../figure-fed42/mcc_vs_fraction.png`.> Jalankan setelah 42_01. JANGAN mengarang angka.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.')); import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
FIGDIR=os.path.join('..','figure-fed42'); os.makedirs(FIGDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEED=42
print('setup 42_03 siap')

## 1. Muat partisi + run_fed

In [ ]:
def need(path):
    assert os.path.exists(path),'PRASYARAT GAGAL: %s - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')),allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')),allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
def predict_with(W,mu,sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5, seed=SEED):
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy(); rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        if len(yk): mu,sd=fed.zscore_fit(Xk)
        else: mu,sd=np.zeros(X.shape[1]),np.ones(X.shape[1])
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd),y=yk,n=len(yk)))
    Wg=fed.init_weights(seed); f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(ROUNDS):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg,cli[k]['X'],cli[k]['y'],epochs=LOCAL_EPOCHS,lr=LR,mu=0.0,W_global=Wg,seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl,gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule,wl,sizes,f=f_byz,beta=max(1,f_byz))
    mu_g,sd_g=fed.zscore_fit(X); pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    return matthews_corrcoef(yg,yp)
RULES=['fedavg','krum','multi_krum','trimmed_mean','median']
print('K',K,'| run_fed siap')

## 2. Sweep MCC vs fraksi x aturan

In [ ]:
# MCC vs fraksi penyerang untuk SEMUA aturan (serangan = label_flip, paling umum).
frac_map={0.0:(),0.1:(0,),0.2:(0,1),0.3:(0,1,2)}
ATTACK='label_flip'
rows=[]
for rule in RULES:
    for frac,mal in frac_map.items():
        atk=None if frac==0.0 else ATTACK
        mcc=run_fed(rule, attack=atk, mal_ids=mal)
        rows.append({'rule':rule,'frac':frac,'MCC':round(mcc,4)})
        print(f'{rule:12s} f={int(frac*100)}% -> MCC {mcc:.4f}')
dfc=pd.DataFrame(rows); dfc.to_csv(os.path.join(OUTDIR,'vuln_curve.csv'),index=False)
import IPython.display as ipd; ipd.display(dfc.pivot(index='rule',columns='frac',values='MCC'))
print('=== SEL 2 (sweep kurva) SELESAI ===')

## 3. Grafik kurva (teks figur Inggris)

In [ ]:
fig,ax=plt.subplots(figsize=(6,3.6))
fr=sorted(frac_map.keys())
for rule in RULES:
    sub=dfc[dfc['rule']==rule].sort_values('frac')
    ax.plot([int(x*100) for x in sub['frac']], sub['MCC'], marker='o', label=rule)
ax.set_xlabel('Attacker fraction (%)'); ax.set_ylabel('MCC (GLOBAL-test)')
ax.set_title('MCC vs attacker fraction by aggregation rule (label-flipping)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(FIGDIR,'mcc_vs_fraction.png'),dpi=150); plt.close()
print('grafik -> ../figure-fed42/mcc_vs_fraction.png')
print('=== SEL 3 (grafik kurva) SELESAI ===')

## 4. Catatan untuk naskah- Gambar `mcc_vs_fraction.png` -> figur RQ1/RQ2 di `paper-4-2.tex` (satu grafik,  menunjukkan FedAvg jatuh sedangkan robust-agg bertahan seiring fraksi naik).- Serangan dipilih label-flipping (umum); varian scale/backdoor ada di 42_04/42_06.- Semua dari `robust_agg.py` + `fed_node.py` -> konsisten.